# 02 Normalize Patients

This notebook transforms raw `patients.csv` into a normalized patient structure:
- `patient.csv`
- `patient_address.csv`

The address is stored separately and linked by a foreign key to keep the data model normalized.

In [1]:
import pandas as pd
import numpy as np
import uuid
from pathlib import Path

RAW_DATA_DIR = Path('data/raw')
PROCESSED_DATA_DIR = Path('data/processed')

PATIENTS_FILE = RAW_DATA_DIR / 'patients.csv'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def normalize_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(r'^\s*$', np.nan, regex=True)


def normalize_gender(value: str) -> str:
    if pd.isna(value):
        return 'UNKNOWN'

    value = str(value).strip().upper()

    if value == 'M':
        return 'MALE'
    if value == 'F':
        return 'FEMALE'

    return 'UNKNOWN'


def generate_patient_number(index: int) -> str:
    return f'P{index:08d}'


def clean_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    string_columns = df.select_dtypes(include=['object', 'string']).columns

    for col in string_columns:
        df[col] = df[col].astype('string').str.strip()

    return df

In [3]:
raw_patients_df = pd.read_csv(PATIENTS_FILE)
print('Raw patients shape:', raw_patients_df.shape)
raw_patients_df.head()

Raw patients shape: (333, 28)


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,2d68ad16-268a-478c-1f84-d0f1976e1a46,2019-09-10,NaN,999-71-2318,NaN,NaN,NaN,Mauro926,Isaias604,Braun514,NaN,NaN,NaN,white,nonhispanic,M,Danvers Massachusetts US,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,25021.0,2186,42.259372,-71.012584,11394.17,0.00,103247
1,5e688e99-61b3-5c88-3f60-21df8aaced27,2009-08-17,NaN,999-87-5260,S99921124,NaN,NaN,Wallace647,Raymond398,Effertz744,NaN,NaN,NaN,native,nonhispanic,M,Southwick Massachusetts US,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,25025.0,2151,42.425852,-70.948221,2941.72,48360.98,22038
2,982750f4-569b-5949-196b-60699bdda3fb,1986-06-01,NaN,999-41-6411,S99976992,X85134600X,Mrs.,Dwana281,Zora492,Tremblay80,NaN,Schuppe920,D,white,nonhispanic,F,Scituate Massachusetts US,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,25017.0,1851,42.665029,-71.344097,346469.31,464266.21,51919
3,6fbddf55-7096-b883-7cd3-260f27953080,1981-01-17,NaN,999-94-1171,S99913042,X45473831X,Mrs.,Clarinda196,Serena400,O'Connell601,NaN,Mayert710,M,white,nonhispanic,F,Southwick Massachusetts US,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,25017.0,1730,42.506505,-71.211440,14876.49,893681.31,18417
4,46ee9d82-b52c-856d-069b-5064ff052225,1996-01-06,NaN,999-44-2741,S99939437,X24298728X,Mrs.,Shalanda398,Kelsie51,Treutel973,NaN,Terry864,M,white,nonhispanic,F,Marlborough Massachusetts US,925 Brown Annex,Lowell,Massachusetts,Middlesex County,25017.0,1854,42.624058,-71.299623,107143.67,137455.64,124612


In [4]:
rename_map = {
    'Id': 'source_patient_id',
    'BIRTHDATE': 'birth_date',
    'DEATHDATE': 'death_date',
    'SSN': 'ssn',
    'DRIVERS': 'drivers_license',
    'PASSPORT': 'passport',
    'PREFIX': 'prefix',
    'FIRST': 'first_name',
    'LAST': 'last_name',
    'SUFFIX': 'suffix',
    'MAIDEN': 'maiden_name',
    'MARITAL': 'marital_status',
    'RACE': 'race',
    'ETHNICITY': 'ethnicity',
    'GENDER': 'gender',
    'BIRTHPLACE': 'birth_place',
    'ADDRESS': 'address_line',
    'CITY': 'city',
    'STATE': 'state',
    'COUNTY': 'county',
    'ZIP': 'zip_code',
    'LAT': 'latitude',
    'LON': 'longitude',
    'HEALTHCARE_EXPENSES': 'healthcare_expenses',
    'HEALTHCARE_COVERAGE': 'healthcare_coverage',
}

patients_df = raw_patients_df.rename(columns=rename_map).copy()
patients_df.head()

,source_patient_id,birth_date,death_date,ssn,drivers_license,passport,prefix,first_name,MIDDLE,last_name,suffix,maiden_name,marital_status,race,ethnicity,gender,birth_place,address_line,city,state,county,FIPS,zip_code,latitude,longitude,healthcare_expenses,healthcare_coverage,INCOME
0,2d68ad16-268a-478c-1f84-d0f1976e1a46,2019-09-10,NaN,999-71-2318,NaN,NaN,NaN,Mauro926,Isaias604,Braun514,NaN,NaN,NaN,white,nonhispanic,M,Danvers Massachusetts US,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,25021.0,2186,42.259372,-71.012584,11394.17,0.00,103247
1,5e688e99-61b3-5c88-3f60-21df8aaced27,2009-08-17,NaN,999-87-5260,S99921124,NaN,NaN,Wallace647,Raymond398,Effertz744,NaN,NaN,NaN,native,nonhispanic,M,Southwick Massachusetts US,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,25025.0,2151,42.425852,-70.948221,2941.72,48360.98,22038
2,982750f4-569b-5949-196b-60699bdda3fb,1986-06-01,NaN,999-41-6411,S99976992,X85134600X,Mrs.,Dwana281,Zora492,Tremblay80,NaN,Schuppe920,D,white,nonhispanic,F,Scituate Massachusetts US,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,25017.0,1851,42.665029,-71.344097,346469.31,464266.21,51919
3,6fbddf55-7096-b883-7cd3-260f27953080,1981-01-17,NaN,999-94-1171,S99913042,X45473831X,Mrs.,Clarinda196,Serena400,O'Connell601,NaN,Mayert710,M,white,nonhispanic,F,Southwick Massachusetts US,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,25017.0,1730,42.506505,-71.211440,14876.49,893681.31,18417
4,46ee9d82-b52c-856d-069b-5064ff052225,1996-01-06,NaN,999-44-2741,S99939437,X24298728X,Mrs.,Shalanda398,Kelsie51,Treutel973,NaN,Terry864,M,white,nonhispanic,F,Marlborough Massachusetts US,925 Brown Annex,Lowell,Massachusetts,Middlesex County,25017.0,1854,42.624058,-71.299623,107143.67,137455.64,124612


In [5]:
required_columns = [
    'source_patient_id',
    'birth_date',
    'death_date',
    'first_name',
    'last_name',
    'marital_status',
    'race',
    'ethnicity',
    'gender',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]

existing_columns = [col for col in required_columns if col in patients_df.columns]
patients_df = patients_df[existing_columns].copy()

print('Selected columns:', patients_df.columns.tolist())
patients_df.head()

Selected columns: ['source_patient_id', 'birth_date', 'death_date', 'first_name', 'last_name', 'marital_status', 'race', 'ethnicity', 'gender', 'address_line', 'city', 'state', 'county', 'zip_code']


,source_patient_id,birth_date,death_date,first_name,last_name,marital_status,race,ethnicity,gender,address_line,city,state,county,zip_code
0,2d68ad16-268a-478c-1f84-d0f1976e1a46,2019-09-10,NaN,Mauro926,Braun514,NaN,white,nonhispanic,M,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,2186
1,5e688e99-61b3-5c88-3f60-21df8aaced27,2009-08-17,NaN,Wallace647,Effertz744,NaN,native,nonhispanic,M,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,2151
2,982750f4-569b-5949-196b-60699bdda3fb,1986-06-01,NaN,Dwana281,Tremblay80,D,white,nonhispanic,F,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,1851
3,6fbddf55-7096-b883-7cd3-260f27953080,1981-01-17,NaN,Clarinda196,O'Connell601,M,white,nonhispanic,F,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,1730
4,46ee9d82-b52c-856d-069b-5064ff052225,1996-01-06,NaN,Shalanda398,Treutel973,M,white,nonhispanic,F,925 Brown Annex,Lowell,Massachusetts,Middlesex County,1854


In [6]:
patients_df = normalize_empty_strings(patients_df)
patients_df = clean_string_columns(patients_df)

if 'birth_date' in patients_df.columns:
    patients_df['birth_date'] = pd.to_datetime(patients_df['birth_date'], errors='coerce').dt.date

if 'death_date' in patients_df.columns:
    patients_df['death_date'] = pd.to_datetime(patients_df['death_date'], errors='coerce').dt.date

if 'first_name' in patients_df.columns:
    patients_df['first_name'] = patients_df['first_name'].str.title()

if 'last_name' in patients_df.columns:
    patients_df['last_name'] = patients_df['last_name'].str.title()

if 'gender' in patients_df.columns:
    patients_df['gender'] = patients_df['gender'].apply(normalize_gender)
else:
    patients_df['gender'] = 'UNKNOWN'

patients_df['deceased'] = patients_df['death_date'].notna()

patients_df['full_name'] = (patients_df['first_name'].fillna('') + ' ' + patients_df['last_name'].fillna('')).str.strip()
patients_df['full_name'] = patients_df['full_name'].replace('', pd.NA)

patients_df = patients_df.drop_duplicates(subset=['source_patient_id']).reset_index(drop=True)

print('Cleaned patients shape:', patients_df.shape)
patients_df.head()

Cleaned patients shape: (333, 16)


,source_patient_id,birth_date,death_date,first_name,last_name,marital_status,race,ethnicity,gender,address_line,city,state,county,zip_code,deceased,full_name
0,2d68ad16-268a-478c-1f84-d0f1976e1a46,2019-09-10,NaT,Mauro926,Braun514,<NA>,white,nonhispanic,MALE,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,2186,False,Mauro926 Braun514
1,5e688e99-61b3-5c88-3f60-21df8aaced27,2009-08-17,NaT,Wallace647,Effertz744,<NA>,native,nonhispanic,MALE,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,2151,False,Wallace647 Effertz744
2,982750f4-569b-5949-196b-60699bdda3fb,1986-06-01,NaT,Dwana281,Tremblay80,D,white,nonhispanic,FEMALE,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,1851,False,Dwana281 Tremblay80
3,6fbddf55-7096-b883-7cd3-260f27953080,1981-01-17,NaT,Clarinda196,O'Connell601,M,white,nonhispanic,FEMALE,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,1730,False,Clarinda196 O'Connell601
4,46ee9d82-b52c-856d-069b-5064ff052225,1996-01-06,NaT,Shalanda398,Treutel973,M,white,nonhispanic,FEMALE,925 Brown Annex,Lowell,Massachusetts,Middlesex County,1854,False,Shalanda398 Treutel973


In [7]:
patient_df = patients_df[[
    'source_patient_id',
    'first_name',
    'last_name',
    'full_name',
    'birth_date',
    'gender',
    'deceased',
    'death_date',
    'marital_status',
    'race',
    'ethnicity',
]].copy()

patient_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(patient_df))])
patient_df.insert(1, 'patient_number', [generate_patient_number(i + 1) for i in range(len(patient_df))])

patient_df.head()

,id,patient_number,source_patient_id,first_name,last_name,full_name,birth_date,gender,deceased,death_date,marital_status,race,ethnicity
0,535e0105-c571-4f45-be53-0ae18f605b0a,P00000001,2d68ad16-268a-478c-1f84-d0f1976e1a46,Mauro926,Braun514,Mauro926 Braun514,2019-09-10,MALE,False,NaT,<NA>,white,nonhispanic
1,77ca1acf-5d58-4574-8ba4-9957c932bc7a,P00000002,5e688e99-61b3-5c88-3f60-21df8aaced27,Wallace647,Effertz744,Wallace647 Effertz744,2009-08-17,MALE,False,NaT,<NA>,native,nonhispanic
2,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,P00000003,982750f4-569b-5949-196b-60699bdda3fb,Dwana281,Tremblay80,Dwana281 Tremblay80,1986-06-01,FEMALE,False,NaT,D,white,nonhispanic
3,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,P00000004,6fbddf55-7096-b883-7cd3-260f27953080,Clarinda196,O'Connell601,Clarinda196 O'Connell601,1981-01-17,FEMALE,False,NaT,M,white,nonhispanic
4,d337e89d-2645-495d-82a3-d971292fd4c4,P00000005,46ee9d82-b52c-856d-069b-5064ff052225,Shalanda398,Treutel973,Shalanda398 Treutel973,1996-01-06,FEMALE,False,NaT,M,white,nonhispanic


In [8]:
patient_address_df = patients_df[[
    'source_patient_id',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]].copy()

patient_address_df = patient_address_df.merge(
    patient_df[['id', 'source_patient_id']],
    on='source_patient_id',
    how='inner'
)

patient_address_df = patient_address_df.rename(columns={'id': 'patient_id'})
patient_address_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(patient_address_df))])

patient_address_df = patient_address_df[[
    'id',
    'patient_id',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]].copy()

address_content_columns = ['address_line', 'city', 'state', 'county', 'zip_code']
patient_address_df = patient_address_df[patient_address_df[address_content_columns].notna().any(axis=1)].reset_index(drop=True)

patient_address_df.head()

,id,patient_id,address_line,city,state,county,zip_code
0,c07beb75-9d75-44f3-bda4-c2ab48d7924a,535e0105-c571-4f45-be53-0ae18f605b0a,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,2186
1,8e9cddd3-3b36-4ace-9718-fbf49733d4ba,77ca1acf-5d58-4574-8ba4-9957c932bc7a,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,2151
2,2891fb01-83b0-479c-aa1e-75711065f32c,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,1851
3,e334abc0-6d5f-4fba-98f9-90e192a6a377,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,1730
4,2acc00d2-1327-479b-906c-d0059b78797c,d337e89d-2645-495d-82a3-d971292fd4c4,925 Brown Annex,Lowell,Massachusetts,Middlesex County,1854


In [9]:
print('Patient table shape:', patient_df.shape)
print('Patient address table shape:', patient_address_df.shape)

print('\nPatient null counts:')
print(patient_df.isna().sum())

print('\nPatient address null counts:')
print(patient_address_df.isna().sum())

print('\nDuplicate source_patient_id in patient table:', patient_df['source_patient_id'].duplicated().sum())
print('Duplicate patient_number in patient table:', patient_df['patient_number'].duplicated().sum())
print('Address records without valid patient_id:', patient_address_df['patient_id'].isna().sum())

Patient table shape: (333, 13)
Patient address table shape: (333, 7)

Patient null counts:
id                     0
patient_number         0
source_patient_id      0
first_name             0
last_name              0
full_name              0
birth_date             0
gender                 0
deceased               0
death_date           300
marital_status       132
race                   0
ethnicity              0
dtype: int64

Patient address null counts:
id              0
patient_id      0
address_line    0
city            0
state           0
county          0
zip_code        0
dtype: int64

Duplicate source_patient_id in patient table: 0
Duplicate patient_number in patient table: 0
Address records without valid patient_id: 0


In [10]:
patient_output_file = PROCESSED_DATA_DIR / 'patient.csv'
patient_address_output_file = PROCESSED_DATA_DIR / 'patient_address.csv'

patient_df.to_csv(patient_output_file, index=False)
patient_address_df.to_csv(patient_address_output_file, index=False)

print('Exported:', patient_output_file)
print('Exported:', patient_address_output_file)

Exported: data\processed\patient.csv
Exported: data\processed\patient_address.csv


In [11]:
display(patient_df.head(10))
display(patient_address_df.head(10))

,id,patient_number,source_patient_id,first_name,last_name,full_name,birth_date,gender,deceased,death_date,marital_status,race,ethnicity
0,535e0105-c571-4f45-be53-0ae18f605b0a,P00000001,2d68ad16-268a-478c-1f84-d0f1976e1a46,Mauro926,Braun514,Mauro926 Braun514,2019-09-10,MALE,False,NaT,<NA>,white,nonhispanic
1,77ca1acf-5d58-4574-8ba4-9957c932bc7a,P00000002,5e688e99-61b3-5c88-3f60-21df8aaced27,Wallace647,Effertz744,Wallace647 Effertz744,2009-08-17,MALE,False,NaT,<NA>,native,nonhispanic
2,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,P00000003,982750f4-569b-5949-196b-60699bdda3fb,Dwana281,Tremblay80,Dwana281 Tremblay80,1986-06-01,FEMALE,False,NaT,D,white,nonhispanic
3,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,P00000004,6fbddf55-7096-b883-7cd3-260f27953080,Clarinda196,O'Connell601,Clarinda196 O'Connell601,1981-01-17,FEMALE,False,NaT,M,white,nonhispanic
4,d337e89d-2645-495d-82a3-d971292fd4c4,P00000005,46ee9d82-b52c-856d-069b-5064ff052225,Shalanda398,Treutel973,Shalanda398 Treutel973,1996-01-06,FEMALE,False,NaT,M,white,nonhispanic
5,30898761-15a4-47cd-9c61-fe9b6ff4d8a1,P00000006,76b20010-c318-5754-8c85-983aa538522f,Jody426,Hickle134,Jody426 Hickle134,2017-10-06,MALE,False,NaT,<NA>,white,nonhispanic
6,8285a3e3-7a04-4413-8278-422379448f47,P00000007,46976cf7-b0bf-be20-39a5-9f425a52886d,Lindsay928,Zieme486,Lindsay928 Zieme486,2007-04-03,FEMALE,False,NaT,<NA>,white,nonhispanic
7,6e4321e9-9f8a-4bd8-a9db-eeebead4c5df,P00000008,62f60bdb-cc5c-8305-b98b-f2b229a55eca,Angel97,Luettgen772,Angel97 Luettgen772,1975-05-11,FEMALE,True,2025-08-05,S,white,nonhispanic
8,9d64ca04-3821-43fb-98d0-fa3f1672d7d4,P00000009,c053e996-a4c4-6c02-e2b6-284227156c67,Sharyn437,Hagenes547,Sharyn437 Hagenes547,2023-04-03,FEMALE,False,NaT,<NA>,black,hispanic
9,dc310d3e-46d0-411c-934e-ee99d1d91361,P00000010,aee7bbe1-0c45-c028-1e62-1f4cdb30c273,Wilfredo622,Fritsch593,Wilfredo622 Fritsch593,2010-07-12,MALE,False,NaT,<NA>,white,nonhispanic


,id,patient_id,address_line,city,state,county,zip_code
0,c07beb75-9d75-44f3-bda4-c2ab48d7924a,535e0105-c571-4f45-be53-0ae18f605b0a,686 Hilpert Annex,Quincy,Massachusetts,Norfolk County,2186
1,8e9cddd3-3b36-4ace-9718-fbf49733d4ba,77ca1acf-5d58-4574-8ba4-9957c932bc7a,788 Hansen Skyway,Revere,Massachusetts,Suffolk County,2151
2,2891fb01-83b0-479c-aa1e-75711065f32c,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,864 Wisozk Skyway Suite 4,Lowell,Massachusetts,Middlesex County,1851
3,e334abc0-6d5f-4fba-98f9-90e192a6a377,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,205 Doyle Parade,Burlington,Massachusetts,Middlesex County,1730
4,2acc00d2-1327-479b-906c-d0059b78797c,d337e89d-2645-495d-82a3-d971292fd4c4,925 Brown Annex,Lowell,Massachusetts,Middlesex County,1854
5,4c4f9488-aa36-4086-b1f3-cceb8b546578,30898761-15a4-47cd-9c61-fe9b6ff4d8a1,368 Marquardt Highlands,Newburyport,Massachusetts,Essex County,1950
6,9a42fec3-e3f4-43d0-88a5-fe5c720d107c,8285a3e3-7a04-4413-8278-422379448f47,794 Dicki Neck Unit 4,Worcester,Massachusetts,Worcester County,1607
7,3461f8a9-79d4-4f52-ad33-e7f7a138ac74,6e4321e9-9f8a-4bd8-a9db-eeebead4c5df,120 Collins Path Suite 22,Fall River,Massachusetts,Bristol County,2790
8,d7e59cbc-e137-477e-b9d8-d2b4224f664d,9d64ca04-3821-43fb-98d0-fa3f1672d7d4,663 Blick Plaza,Milford,Massachusetts,Worcester County,1757
9,ee8366ad-49cf-4f31-a5cc-33adb26a7cce,dc310d3e-46d0-411c-934e-ee99d1d91361,265 Schamberger Rapid Unit 70,Whitman,Massachusetts,Plymouth County,0
